In [ ]:
# Core
import pandas as pd
import numpy as np
import re
import warnings
import os
warnings.filterwarnings("ignore")

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#Stopwords
!pip install stop-words
from stop_words import get_stop_words

#In case colab crashes because of fasttext saying numpy version mis-match (otherwise install normally)
!pip -q uninstall -y fasttext numpy
!pip -q install "numpy==1.26.4"
!pip -q install fasttext


In [ ]:
print("numpy:", np.__version__)


In [ ]:
def pick_file(clean_path, raw_path):
    return clean_path if os.path.exists(clean_path) else raw_path

train_path = pick_file("/content/Train_clean.csv", "/content/Train.csv")
valid_path = pick_file("/content/Valid_clean.csv", "/content/Valid.csv")
test_path  = pick_file("/content/Test_clean.csv",  "/content/Test.csv")

train_df = pd.read_csv(train_path, engine="python")
valid_df = pd.read_csv(valid_path, engine="python")
test_df  = pd.read_csv(test_path,  engine="python")

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

train_df.head()


In [ ]:
def basic_checks(df, name):
    print(f"\n--- {name} ---")
    print("Columns:", list(df.columns))
    print("Missing values:\n", df.isna().sum())
    if "label" in df.columns:
        print("Label counts:\n", df["label"].value_counts(dropna=False))
    if "text" in df.columns:
        print("Empty/blank texts:", (df["text"].astype(str).str.strip() == "").sum())

basic_checks(train_df, "TRAIN")
basic_checks(valid_df, "VALID")
basic_checks(test_df,  "TEST")


In [ ]:
def leakage_report(train, valid, test):
    train_set = set(train["text"].astype(str))
    valid_set = set(valid["text"].astype(str))
    test_set  = set(test["text"].astype(str))

    tv = len(train_set & valid_set)
    tt = len(train_set & test_set)
    vt = len(valid_set & test_set)

    print("Overlap Train ∩ Valid:", tv)
    print("Overlap Train ∩ Test :", tt)
    print("Overlap Valid ∩ Test :", vt)

leakage_report(train_df, valid_df, test_df)


In [ ]:
def length_stats(df, name):
    lengths = df["text"].astype(str).str.len()
    words = df["text"].astype(str).str.split().apply(len)
    print(f"\n--- {name} length stats ---")
    print("Chars  : min/median/mean/95%/max =", int(lengths.min()), int(lengths.median()), int(lengths.mean()), int(lengths.quantile(0.95)), int(lengths.max()))
    print("Words  : min/median/mean/95%/max =", int(words.min()), int(words.median()), int(words.mean()), int(words.quantile(0.95)), int(words.max()))

length_stats(train_df, "TRAIN")
length_stats(valid_df, "VALID")
length_stats(test_df,  "TEST")


In [ ]:
def text_artifacts(df, name, n=5):
    texts = df["text"].astype(str)
    has_html = texts.str.contains(r"<br\s*/?>|</?[^>]+>", regex=True).mean()
    has_quotes = texts.str.contains(r'"').mean()
    has_non_ascii = texts.str.contains(r"[^\x00-\x7F]+", regex=True).mean()
    print(f"\n--- {name} artifacts ---")
    print("Fraction with HTML tags:", round(has_html, 4))
    print("Fraction with quotes:", round(has_quotes, 4))
    print("Fraction with non-ascii:", round(has_non_ascii, 4))
    html_examples = df[texts.str.contains(r"<br\s*/?>|</?[^>]+>", regex=True)].head(n)["text"].tolist()
    if html_examples:
        print("\nExample HTML texts:")
        for i, t in enumerate(html_examples, 1):
            print(f"\n[{i}]\n{t[:400]}")

text_artifacts(train_df, "TRAIN")


In [ ]:
def label_balance(df, name):
    counts = df["label"].value_counts()
    total = counts.sum()
    print(f"\n--- {name} balance ---")
    for k, v in counts.items():
        print(k, ":", v, f"({v/total:.2%})")

label_balance(train_df, "TRAIN")
label_balance(valid_df, "VALID")
label_balance(test_df,  "TEST")


In [ ]:
STOP_WORDS = set(get_stop_words("en"))
def preprocess(text, remove_stopwords=False):
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"</?[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    if remove_stopwords:
        tokens = tokens = [t for t in tokens if t not in STOP_WORDS]
    return " ".join(tokens)


In [ ]:
Xtr = train_df["text"].apply(lambda x: preprocess(x, False))
Xva = valid_df["text"].apply(lambda x: preprocess(x, False))
Xte = test_df["text"].apply(lambda x: preprocess(x, False))

ytr = train_df["label"]
yva = valid_df["label"]
yte = test_df["label"]

ytr = ytr.map({0:0, 1:1}) if ytr.dtype != object else ytr.map({"negative":0,"positive":1})
yva = yva.map({0:0, 1:1}) if yva.dtype != object else yva.map({"negative":0,"positive":1})
yte = yte.map({0:0, 1:1}) if yte.dtype != object else yte.map({"negative":0,"positive":1})

vec1 = TfidfVectorizer(ngram_range=(1,2), max_features=20000)
Xtr_v = vec1.fit_transform(Xtr)
Xva_v = vec1.transform(Xva)
Xte_v = vec1.transform(Xte)

clf1 = LogisticRegression(max_iter=2000)
clf1.fit(Xtr_v, ytr)

va_pred1 = clf1.predict(Xva_v)
te_pred1 = clf1.predict(Xte_v)


In [ ]:
print("Model 1 – Test")
print("Accuracy :", accuracy_score(yte, te_pred1))
print("Precision:", precision_score(yte, te_pred1))
print("Recall   :", recall_score(yte, te_pred1))
print("F1       :", f1_score(yte, te_pred1))


In [ ]:
Xtr = train_df["text"].apply(lambda x: preprocess(x, True))
Xva = valid_df["text"].apply(lambda x: preprocess(x, True))
Xte = test_df["text"].apply(lambda x: preprocess(x, True))

vec2 = TfidfVectorizer(ngram_range=(1,2), max_features=20000)
Xtr_v = vec2.fit_transform(Xtr)
Xva_v = vec2.transform(Xva)
Xte_v = vec2.transform(Xte)

clf2 = LogisticRegression(max_iter=2000)
clf2.fit(Xtr_v, ytr)

te_pred2 = clf2.predict(Xte_v)


In [ ]:
print("Model 2 – Test")
print("Accuracy :", accuracy_score(yte, te_pred2))
print("Precision:", precision_score(yte, te_pred2))
print("Recall   :", recall_score(yte, te_pred2))
print("F1       :", f1_score(yte, te_pred2))


In [ ]:
def preprocess_ft(text):
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"</?[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def to_fasttext_safe(df, path):
    kept, dropped = 0, 0
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            label = "__label__positive" if row["label"] in [1, "positive"] else "__label__negative"
            text = preprocess_ft(row["text"])
            if len(text.split()) < 2:
                dropped += 1
                continue
            f.write(f"{label} {text}\n")
            kept += 1
    print(path, "kept:", kept, "| dropped:", dropped)

to_fasttext_safe(train_df, "train_ft.txt")
to_fasttext_safe(valid_df, "valid_ft.txt")
to_fasttext_safe(test_df,  "test_ft.txt")


In [ ]:
def eval_fasttext(model, file_path):
    y_true, y_pred = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(" ", 1)
            if len(parts) != 2:
                continue
            true_label, text = parts
            pred = model.predict(text.strip(), k=1)[0][0]
            y_true.append(1 if true_label == "__label__positive" else 0)
            y_pred.append(1 if pred == "__label__positive" else 0)
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred)


In [ ]:
param_grid = {
    "epoch": [5, 10, 20, 30],
    "lr": [0.05, 0.1, 0.2, 0.3],
    "wordNgrams": [1, 2],
    "dim": [50, 100],
    "minCount": [2]
}

total = (
    len(param_grid["epoch"]) *
    len(param_grid["lr"]) *
    len(param_grid["wordNgrams"]) *
    len(param_grid["dim"]) *
    len(param_grid["minCount"])
)
print("Total configs:", total)


In [ ]:
import fasttext
import pandas as pd
import gc

results = []
run_i = 0
#can set to any other number in case colab crashes for eg something like 16
MAX_RUNS = None

for epoch in param_grid["epoch"]:
    for lr in param_grid["lr"]:
        for ngrams in param_grid["wordNgrams"]:
            for dim in param_grid["dim"]:
                for minCount in param_grid["minCount"]:

                    run_i += 1
                    if MAX_RUNS is not None and run_i > MAX_RUNS:
                        break

                    model = None

                    try:
                        model = fasttext.train_supervised(
                            input="train_ft.txt",
                            epoch=epoch,
                            lr=lr,
                            wordNgrams=ngrams,
                            dim=dim,
                            minCount=minCount,
                            loss="softmax",
                            verbose=0
                        )

                        acc, f1 = eval_fasttext(model, "valid_ft.txt")

                        results.append({
                            "epoch": epoch,
                            "lr": lr,
                            "wordNgrams": ngrams,
                            "dim": dim,
                            "minCount": minCount,
                            "valid_acc": acc,
                            "valid_f1": f1
                        })

                    except Exception as e:
                        results.append({
                            "epoch": epoch,
                            "lr": lr,
                            "wordNgrams": ngrams,
                            "dim": dim,
                            "minCount": minCount,
                            "valid_acc": None,
                            "valid_f1": None,
                            "error": str(e)
                        })

                    finally:
                        if model is not None:
                            del model
                        gc.collect()

                    if run_i % 4 == 0:
                        print(f"Completed {run_i} / {total}")

            if MAX_RUNS is not None and run_i >= MAX_RUNS:
                break
        if MAX_RUNS is not None and run_i >= MAX_RUNS:
            break
    if MAX_RUNS is not None and run_i >= MAX_RUNS:
        break

results_df = pd.DataFrame(results)
results_df_sorted = results_df.dropna(subset=["valid_f1"]).sort_values("valid_f1", ascending=False)

print("Successful runs:", len(results_df_sorted), "/", len(results_df))
results_df_sorted.head(10)


In [ ]:
best_cfg = {
    "epoch": 20,
    "lr": 0.10,
    "wordNgrams": 2,
    "dim": 100,
    "minCount": 2
}

best_model = fasttext.train_supervised(
    input="train_ft.txt",   # full training set file
    epoch=best_cfg["epoch"],
    lr=best_cfg["lr"],
    wordNgrams=best_cfg["wordNgrams"],
    dim=best_cfg["dim"],
    minCount=best_cfg["minCount"],
    loss="softmax",
    verbose=0
)

test_acc, test_f1 = eval_fasttext(best_model, "test_ft.txt")

print("Best FastText config:", best_cfg)
print("TEST accuracy:", test_acc)
print("TEST F1      :", test_f1)


In [ ]:
fasttext_best_cfg = {'epoch': 20, 'lr': 0.1, 'wordNgrams': 2, 'dim': 100, 'minCount': 2}
fasttext_test = {"accuracy": 0.9062, "f1": 0.907293931607037}
fasttext_best_cfg, fasttext_test


In [ ]:
## Here just to get the numbers at the same place (still the same one as before)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def eval_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

ytr = train_df["label"]
yva = valid_df["label"]
yte = test_df["label"]

if ytr.dtype == object:
    ytr = ytr.map({"negative":0, "positive":1})
    yva = yva.map({"negative":0, "positive":1})
    yte = yte.map({"negative":0, "positive":1})

def run_tfidf_lr(remove_stopwords):
    Xtr = train_df["text"].apply(lambda x: preprocess(x, remove_stopwords))
    Xva = valid_df["text"].apply(lambda x: preprocess(x, remove_stopwords))
    Xte = test_df["text"].apply(lambda x: preprocess(x, remove_stopwords))

    vec = TfidfVectorizer(ngram_range=(1,2), max_features=20000)
    Xtr_v = vec.fit_transform(Xtr)
    Xva_v = vec.transform(Xva)
    Xte_v = vec.transform(Xte)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr_v, ytr)

    va_pred = clf.predict(Xva_v)
    te_pred = clf.predict(Xte_v)

    return eval_metrics(yva, va_pred), eval_metrics(yte, te_pred)

m1_valid, m1_test = run_tfidf_lr(remove_stopwords=False)
m2_valid, m2_test = run_tfidf_lr(remove_stopwords=True)

m1_test, m2_test


In [ ]:
#Here for precision and recall
from sklearn.metrics import precision_score, recall_score

def eval_fasttext_full(model, file_path):
    y_true, y_pred = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            true_label, text = line.split(" ", 1)
            pred = model.predict(text.strip(), k=1)[0][0]
            y_true.append(1 if true_label == "__label__positive" else 0)
            y_pred.append(1 if pred == "__label__positive" else 0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

ft_test_full = eval_fasttext_full(best_model, "test_ft.txt")
ft_test_full


In [ ]:
'''
## Uncomment this if you want to have the results of all 3 models in a table side by side


final_results = pd.DataFrame([
    ["TF-IDF + LR (no stopwords)", m1_test["accuracy"], m1_test["precision"], m1_test["recall"], m1_test["f1"]],
    ["TF-IDF + LR (with stopwords)", m2_test["accuracy"], m2_test["precision"], m2_test["recall"], m2_test["f1"]],
    ["FastText (tuned on valid)", ft_test_full["accuracy"], ft_test_full["precision"], ft_test_full["recall"], ft_test_full["f1"]],
], columns=["Model", "Accuracy", "Precision", "Recall", "F1"])

final_results
'''
